# Amazon ML Challenge 2026 — Track 03: Scalable Candidate Generation (Blocking)
**Objective:** Reduce the combinatorial comparison space from $O(N \times M)$ (~22.4 Trillion pairs) to a high-recall candidate set.
- 1. Country-partitioned candidate filtering
- 2. Standard rule-based blocking keys
- 3. Inverted index token blocking with IDF weighting
- 4. Candidate count distribution per query (mean, median, p90, max)
- 5. Generating candidate pairs for downstream matching


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import src

print("Blocking modules loaded successfully!")


## 1. Candidate Generation Strategy
Because a Cartesian join between Source 1 (2.2M) and Sources 2+3 (10.3M) contains over **22 Trillion comparisons**, we employ:
1. **Partitioning by Country:** Candidates must match the query country (`US`, `IN`, `FR`).
2. **Token Inverted Index with IDF:** Index rare, discriminative business name tokens.
3. **Stop-Token Frequency Cap:** Prune ubiquitous tokens (`group`, `services`, `enterprises`) that explode candidate list size without providing selectivity.
4. **Top-K Candidate Budget:** Rank candidate matches by IDF-weighted token overlap and cap at $K=30$ candidates per query.


In [ ]:
# Load training sample for blocking demonstration
print("Loading data samples...")
s1_df = src.load_source("train", 1, nrows=10000)
s2_df = src.load_source("train", 2, nrows=50000)
s3_df = src.load_source("train", 3, nrows=50000)

targets_df = pd.concat([s2_df, s3_df], ignore_index=True)
print(f"Query set (S1): {len(s1_df):,} | Target candidate pool (S2+S3): {len(targets_df):,}")


In [ ]:
# Initialize and fit the Token Inverted Index Blocker
blocker = src.TokenInvertedIndexBlocker(max_token_freq=500, min_token_len=3, top_k=25)
print("Building inverted index on candidate pool...")
blocker.fit(targets_df)
print(f"Inverted index built with {len(blocker.index):,} unique (country, token) posting keys.")


In [ ]:
# Generate candidate pairs for queries
print("Generating candidate pairs for S1 queries...")
candidates_df = blocker.generate_candidates(s1_df)
print(f"Generated {len(candidates_df):,} candidate pairs.")
display(candidates_df.head(10))


## 2. Candidate Count Distribution Analysis
Analyzing the number of candidates generated per Source 1 entity to ensure balanced workload and prevent skew.


In [ ]:
dist_stats = src.evaluate_candidate_distribution(candidates_df)
print("=== CANDIDATE COUNT DISTRIBUTION METRICS ===")
for k, v in dist_stats.items():
    print(f"{k:30s}: {v}")

counts = candidates_df.groupby("source1_entity_id")["candidate_entity_id"].count()
plt.figure(figsize=(8, 4))
plt.hist(counts, bins=25, color="#10b981", edgecolor="black")
plt.title("Distribution of Candidate Count per Query Entity")
plt.xlabel("Number of Candidates")
plt.ylabel("Number of S1 Queries")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Save candidate pairs sample to outputs/
output_path = PROJECT_ROOT / "outputs" / "sample_candidate_pairs.csv"
candidates_df.head(1000).to_csv(output_path, index=False)
print(f"Sample candidate pairs exported to: {output_path}")
